# 图的遍历

DFS 用栈追踪搜索路径。BFS 用队列按距离逐层访问节点。把已发现节点记录下来可以处理有环图。

In [ ]:
from collections import deque

def dfs_order(graph, start):
    stack = [start]
    seen = set()
    order = []
    while stack:
        node = stack.pop()
        if node in seen:
            continue
        seen.add(node)
        order.append(node)
        stack.extend(reversed(graph[node]))
    return order


def bfs_order(graph, start):
    queue = deque([start])
    seen = {start}
    order = []
    while queue:
        node = queue.popleft()
        order.append(node)
        for neighbor in graph[node]:
            if neighbor not in seen:
                seen.add(neighbor)
                queue.append(neighbor)
    return order


graph = [[1, 2], [0, 3], [0], [1]]
assert dfs_order(graph, 0) == [0, 1, 3, 2]
assert bfs_order(graph, 0) == [0, 1, 2, 3]

## 拓扑排序 Topological Sorting
一种对**有向无环图**的顶点进行进行线性排序的方法  

Kahn法  
1. 创建入度数组 上完一门课 后置课程的入度减一  当且仅当入度为0时加入
2. 同一时间加入队列的会形成平行路径 我们可以使用 `for _ in range(len(q))` 处理同一时刻的状况
3. 按照这样的逻辑 每个节点只会被枚举`一次` 我们只需要在 `popleft()` 之后记录结果即可

In [ ]:
# kahn算法 n个节点的邻接表
from collections import deque

def topologicalSortingKahn(graph):
    indegrees = [0] * len(graph)
    for u in graph:
        for v in u:
            indegrees[v] += 1

    q = deque([i for i, indegree in enumerate(indegrees) if indegree==0])
    ans = []
    while q:
        u = q.popleft()
        ans.append(u)
        for v in graph[u]:
            indegrees[v] -= 1
            if indegrees[v] == 0:  # 当入度为0时马上开始
                q.append(v)

    if len(indegrees) != len(ans):
        return []
    else:
        return ans

In [ ]:
# 课程表 返回是否能上网所有的课
from collections import deque

class Solution:
    def canFinish(self, numCourses: int, prerequisites) -> bool:

        indegrees = [0] * numCourses
        adjacency = [[] for _ in range(numCourses)]
        q = deque()

        for cur, pre in prerequisites:
            indegrees[cur] += 1          # 入度(先修课数量)
            adjacency[pre].append(cur)   # 统计路径 

        for i in range(len(indegrees)):
            if not indegrees[i]: q.append(i)
        
        while q:
            
            pre = q.popleft()
            numCourses -= 1
            for cur in adjacency[pre]:
                indegrees[cur] -= 1      # 上了其中一门课 
                if not indegrees[cur]: q.append(cur)  # 如果前置课程全部上完
        
        return not numCourses

In [ ]:
# 课程表Ⅱ 返回上课的顺序
from collections import deque

class Solution:
    def findOrder(self, numCourses: int, prerequisites) -> list[int]:

        indegrees = [0] * numCourses
        adjacency = [[] for _ in range(numCourses)]
        q = deque()

        for cur, pre in prerequisites:
            indegrees[cur] += 1          # 入度(先修课数量)
            adjacency[pre].append(cur)   # 统计路径 

        for i in range(len(indegrees)):
            if not indegrees[i]: q.append(i)
        
        ans = []
        while q:
            pre = q.popleft()
            numCourses -= 1
            ans.append(pre)
            for cur in adjacency[pre]:
                indegrees[cur] -= 1      # 上了其中一门课 
                if not indegrees[cur]: q.append(cur)  # 如果前置课程全部上完
    
        return ans if not numCourses else []

In [ ]:
n = 3
cache = [2, 3, 7]
dependencies = ["0 0 0", "0 0 0", "1 1 0"]
indegrees = [0] * n
g = [[] for _ in range(n)]

for cur in range(n):
    for pre, symbol in enumerate(dependencies[cur].split()):
        if symbol == '1':
            indegrees[cur] += 1
            g[pre].append(cur)

from collections import deque
q = deque([i for i, indegree in enumerate(indegrees) if indegree == 0])

ans = 0
while q:
    res = 0
    for _ in range(len(q)):
        u = q.popleft()
        res += cache[u]

        for v in g[u]:
            indegrees[v] -= 1
            if indegrees[v] == 0:
                q.append(v)
                
    ans = max(ans, res)

assert ans == 7

In [ ]:
# 并行课程Ⅲ
class Solution:
    def minimumTime(self, n: int, relations, time) -> int:
        indegrees = [0] * n
        g = [[] for i in range(n)]
        for pre, cur in relations:
            pre, cur = map(lambda x:x-1, (pre, cur))
            g[pre].append(cur)
            indegrees[cur] += 1
        
        q = deque([i for i, indegree in enumerate(indegrees) if indegree==0])

        ans = time.copy()
        while q:
            u = q.popleft()
            for v in g[u]:
                # 枚举的顺序不是实际上先修课完成的时间顺序 
                # 所以需要取最大值 维护每门课程开始的最晚时间
                ans[v] = max(ans[v], ans[u]+time[v])
                indegrees[v] -= 1
                if indegrees[v] == 0:
                    q.append(v)

        return max(ans)

## 网格中的最短步数

每次只能走到相邻的可通行格子。边权相同，第一次从队列到达终点时，得到的就是最短步数。

In [ ]:
def grid_distance(grid, start, end):
    if not grid or not grid[0]:
        return None
    height, width = len(grid), len(grid[0])

    def open_cell(row, col):
        return 0 <= row < height and 0 <= col < width and grid[row][col] == 0

    if not open_cell(*start) or not open_cell(*end):
        return None
    queue = deque([(start, 0)])
    visited = {start}
    while queue:
        (row, col), distance = queue.popleft()
        if (row, col) == end:
            return distance
        for dr, dc in ((0, 1), (1, 0), (0, -1), (-1, 0)):
            neighbor = row + dr, col + dc
            if neighbor not in visited and open_cell(*neighbor):
                visited.add(neighbor)
                queue.append((neighbor, distance + 1))
    return None


grid = [[0, 0, 0], [0, 1, 0], [0, 1, 0]]
assert grid_distance(grid, (2, 0), (2, 2)) == 6
assert grid_distance(grid, (0, 0), (0, 0)) == 0
assert grid_distance([[0, 1, 0]], (0, 0), (0, 2)) is None